# Test: append_training_orders pipeline

Runs the pipeline functions against a throwaway **in-memory** DuckDB with dummy data.
Nothing touches `dev.db`, ODS, or amslib.

Put this notebook in `notebooks/` so the relative path to `src/` below works.

In [ ]:
import importlib.util
import logging
from pathlib import Path

import duckdb
import pandas as pd

logging.basicConfig(level=logging.INFO, format='%(levelname)s - %(message)s', force=True)

# load the pipeline file directly, so nothing else in src/ gets imported
PIPELINE_PATH = Path('../src/pipelines/append_training_orders/pipeline.py')
spec = importlib.util.spec_from_file_location('append_pipeline', PIPELINE_PATH)
p = importlib.util.module_from_spec(spec)
spec.loader.exec_module(p)

## A stand-in for the amslib `Tables` object

The pipeline only calls `tables.select(sql)` and `tables.execute(sql)`,
so a tiny wrapper around a DuckDB connection is enough to test it.

In [ ]:
class FakeTables:
    def __init__(self, con):
        self.con = con

    def select(self, sql):
        return self.con.execute(sql).df()

    def execute(self, sql):
        self.con.execute(sql)


con = duckdb.connect()   # in-memory, gone when the kernel restarts
tables = FakeTables(con)

def show(table):
    return con.execute(f'select * from {table}').df()

## Create the tables

Simplified versions of the live tables, plus the training tables as they are in
`app.admin.ddl.sql` (keys and constraints kept, `target_rec_*` audit columns left out).

In [ ]:
con.execute('''
-- live tables (simplified)
create table orders (
    order_id           bigint primary key,
    wr_id              varchar,
    fa_id              varchar,
    wr_status_upd_dttm timestamp,
    comments           varchar,
    instructions       varchar
);

create table cleaned_text (
    order_id             bigint primary key,
    cleaned_comments     varchar,
    cleaned_instructions varchar
);

create sequence seq_category_id start 1;
create table category (
    category_id bigint primary key default nextval('seq_category_id'),
    code        varchar not null unique,
    name        varchar not null
);

-- training tables (from the DDL)
create sequence seq_training_order_id start 1;
create table training_orders (
    training_order_id  bigint primary key default nextval('seq_training_order_id'),
    wr_id              varchar not null,
    fa_id              varchar not null,
    wr_status_upd_dttm timestamp,
    comments           varchar not null,
    instructions       varchar not null,
    constraint uq_order_business_key unique (wr_id, fa_id)
);

create table training_cleaned_text (
    training_order_id    bigint primary key references training_orders(training_order_id),
    cleaned_comments     varchar not null,
    cleaned_instructions varchar not null
);

create table training_order_category (
    training_order_id bigint not null references training_orders(training_order_id),
    category_id       bigint not null references category(category_id),
    manual_label      int not null check (manual_label in (1, 0)),
    primary key (training_order_id, category_id)
);
''')

## Dummy data

Built to hit the edge cases:
- **WR 100** has two field activities (A and B), so it should become two training rows
- **WR 300** has NULL instructions, so it should be stored as `''`, not dropped
- **WR 400 / C** appears twice in `orders`; only the latest should be copied
- **WR 500** has no cleaned text yet, so it should be skipped this run

In [ ]:
con.execute('''
insert into orders values
    (1, '100', 'A', '2026-09-25 08:00', 'replaced meter',       'check meter'),
    (2, '100', 'B', '2026-09-25 08:05', 'meter ok',             'verify reading'),
    (3, '200', 'A', '2026-09-25 08:10', 'battery low replaced', 'battery swap'),
    (4, '300', 'A', '2026-09-25 08:15', 'lcd blank',            NULL),
    (5, '400', 'C', '2026-09-25 08:20', 'first visit',          'inspect'),
    (6, '400', 'C', '2026-09-25 08:40', 'second visit',         'inspect again'),
    (7, '500', 'A', '2026-09-25 08:45', 'not cleaned yet',      'n/a');

insert into cleaned_text values
    (1, 'replac meter',       'check meter'),
    (2, 'meter ok',           'verifi read'),
    (3, 'batteri low replac', 'batteri swap'),
    (4, 'lcd blank',          NULL),
    (5, 'first visit',        'inspect'),
    (6, 'second visit',       'inspect again');
    -- order 7 (WR 500) intentionally has no cleaned text

insert into category (code, name) values
    ('MC',  'METERCHANGE'),
    ('BAT', 'BATTERY');
''')
show('orders')

## Run 1: append everything new

In [ ]:
p.append_training_orders(tables, n=100)
p.append_training_text(tables)
show('training_orders')

In [ ]:
show('training_cleaned_text')

**Check:** 5 rows. WR 100 twice (A and B), WR 300 with `''` instructions,
WR 400 / C once (the 08:40 "second visit"), and no WR 500.

## Run 2: nothing new, so nothing should be added

In [ ]:
p.append_training_orders(tables, n=100)
p.append_training_text(tables)
con.execute('select count(*) from training_orders').fetchone()[0]

## Pretend Nate's labeling pipeline labeled a few orders

In [ ]:
con.execute('''
insert into training_order_category values
    (1, 1, 1),   -- WR 100/A: MC  = yes
    (1, 2, 0),   -- WR 100/A: BAT = no
    (3, 2, 1),   -- WR 200/A: BAT = yes
    (4, 1, 0);   -- WR 300/A: MC  = no
''')
p.build_training_dataset(tables)
show('training_dataset')

**Check:** one column per category (`MC`, `BAT`). `null` = not labeled yet.

## Add a new category: it should appear as a column automatically

In [ ]:
con.execute("insert into category (code, name) values ('LCD', 'BAD_LCD')")
p.build_training_dataset(tables)
show('training_dataset')

## New orders arrive (and WR 500 finally gets cleaned)

In [ ]:
con.execute('''
insert into orders values
    (8, '600', 'A', '2026-09-25 09:00', 'new order', 'swap meter');
insert into cleaned_text values
    (7, 'not clean yet', 'n/a'),
    (8, 'new order',     'swap meter');
''')

p.append_training_orders(tables, n=100)
p.append_training_text(tables)
p.build_training_dataset(tables)
show('training_dataset')

**Check:** WR 500 and WR 600 added with null labels; earlier labels untouched.

## Try the `n` cap

Rerun from the top, but change `n=100` in Run 1 to `n=2`. Only the 2 oldest orders
should be copied, and the rest come in on the next run.